# Garment AI Training Notebook

This notebook prepares a garment-part dataset, optionally preannotates it, trains YOLO, and exports `best.pt` for the Flask app.

Expected classes:

- `neck_label`
- `button`
- `button_placket`
- `zipper`
- `drawcord`
- `eyelet`
- `pocket`
- `kangaroo_pocket`
- `chest_graphic`
- `embroidery_patch`
- `collar`
- `cuff`
- `hem_band`
- `hood`
- `waistband`


## 1. Mount Drive

Put your repo or dataset zip in Google Drive first.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


## 2. Configure Paths

Pick one of these:

- set `REPO_ZIP` if your whole repo is zipped in Drive
- set `REPO_DIR_IN_DRIVE` if the repo folder is already in Drive
- set `RAW_IMAGES_DIR` to a folder of your garment photos


In [ ]:
from pathlib import Path

REPO_ZIP = ''
REPO_DIR_IN_DRIVE = ''
RAW_IMAGES_DIR = ''
WORKDIR = Path('/content/fabric_inventory')
DATASET_DIR = WORKDIR / 'training' / 'garment_dataset'
WEIGHTS_OUT = WORKDIR / 'training' / 'weights'


## 3. Load Repo Into Colab


In [ ]:
import os
import shutil
import zipfile

if WORKDIR.exists():
    shutil.rmtree(WORKDIR)

if REPO_ZIP:
    with zipfile.ZipFile(REPO_ZIP, 'r') as zf:
        zf.extractall('/content')
    extracted = [p for p in Path('/content').iterdir() if p.is_dir() and p.name != 'drive']
    repo_candidates = [p for p in extracted if (p / 'app').exists() and (p / 'scripts').exists()]
    if not repo_candidates:
        raise RuntimeError('Could not find extracted repo folder.')
    shutil.move(str(repo_candidates[0]), str(WORKDIR))
elif REPO_DIR_IN_DRIVE:
    shutil.copytree(REPO_DIR_IN_DRIVE, WORKDIR)
else:
    raise RuntimeError('Set REPO_ZIP or REPO_DIR_IN_DRIVE first.')

os.chdir(WORKDIR)
print('Repo ready at:', WORKDIR)


## 4. Install Dependencies


In [ ]:
!pip install -q -r requirements-garment-ai.txt PyYAML Pillow


## 5. Build Dataset Skeleton From Your Raw Images

Run this if you have a folder of product photos and want the notebook to create `train/val` folders.


In [ ]:
if RAW_IMAGES_DIR:
    !python scripts/prepare_garment_dataset.py --source "{RAW_IMAGES_DIR}" --output "{DATASET_DIR}" --copy
else:
    print('Skipping dataset build because RAW_IMAGES_DIR is empty.')


## 6. Optional: Preannotate Starter Labels

This uses the repo's existing template logic to create starter boxes. You still need to review and correct them.


In [ ]:
RUN_PREANNOTATION = False

if RUN_PREANNOTATION:
    !python scripts/preannotate_garment_dataset.py --data "{DATASET_DIR / 'dataset.yaml'}" --report "training/preannotation_report.json"
else:
    print('Skipping preannotation.')


## 7. Inspect Dataset


In [ ]:
from pathlib import Path

train_images = list((DATASET_DIR / 'images' / 'train').glob('*'))
val_images = list((DATASET_DIR / 'images' / 'val').glob('*'))
train_labels = list((DATASET_DIR / 'labels' / 'train').glob('*.txt'))
val_labels = list((DATASET_DIR / 'labels' / 'val').glob('*.txt'))

print('Train images:', len(train_images))
print('Val images:', len(val_images))
print('Train labels:', len(train_labels))
print('Val labels:', len(val_labels))
print('dataset.yaml exists:', (DATASET_DIR / 'dataset.yaml').exists())


## 8. Train YOLO

Change epochs or batch size if needed.


In [ ]:
EPOCHS = 80
IMGSZ = 960
BATCH = 8
DEVICE = 0
MODEL = 'yolo11n.pt'

!python scripts/train_garment_detector.py --data "{DATASET_DIR / 'dataset.yaml'}" --model "{MODEL}" --epochs {EPOCHS} --imgsz {IMGSZ} --batch {BATCH} --device {DEVICE}


## 9. Copy Best Weights Into App Location


In [ ]:
import shutil

best_weights = WORKDIR / 'runs' / 'detect' / 'garment_parts' / 'weights' / 'best.pt'
WEIGHTS_OUT.mkdir(parents=True, exist_ok=True)
target_weights = WEIGHTS_OUT / 'best.pt'

if not best_weights.exists():
    raise FileNotFoundError(best_weights)

shutil.copy2(best_weights, target_weights)
print('Copied weights to:', target_weights)


## 10. Save Back To Drive

This copies the trained weights somewhere easy to download.


In [ ]:
DRIVE_EXPORT_DIR = '/content/drive/MyDrive/garment_ai_exports'

!mkdir -p "{DRIVE_EXPORT_DIR}"
!cp "{target_weights}" "{DRIVE_EXPORT_DIR}/best.pt"
print('Exported to:', f'{DRIVE_EXPORT_DIR}/best.pt')


## 11. Optional Test Prediction


In [ ]:
TEST_IMAGE = ''

if TEST_IMAGE:
    !python scripts/predict_garment_detector.py --weights "{target_weights}" --image "{TEST_IMAGE}" --output /content/prediction.json --device {DEVICE}
    !cat /content/prediction.json
else:
    print('Set TEST_IMAGE to run a prediction test.')
